# Textile Market Intelligence Agent

## Features
- Competitor research
- Product intelligence extraction
- Strategy recommendations
- AI battlecard reports

## Tech Stack
- LangGraph
- LangChain
- Groq LLM
- Tavily Search
- Python

## Workflow
Research → Intelligence → Strategy → Battlecard



In [1]:
!pip install -U langgraph langchain langchain-groq tavily-python streamlit plotly pandas python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 35.2 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully u

In [2]:
import os

os.environ["GROQ_API_KEY"] = "************"

os.environ["TAVILY_API_KEY"] = "****************"

In [3]:


from typing import TypedDict

from langgraph.graph import StateGraph, END

from langchain_groq import ChatGroq

from tavily import TavilyClient

import ast

In [4]:


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

In [5]:


class MarketState(TypedDict):

    product: str

    competitors: list

    product_intelligence: dict

    strategy_recommendations: str

In [6]:


def competitor_research_node(state):

    product = state["product"]

    search = client.search(
        query=f"""
        top established brands for {product} in India
        only brands older than 3 years
        """,

        search_depth="advanced",

        max_results=5
    )

    content = "\n".join(
        [
            r["content"]

            for r in search["results"]
        ]
    )

    prompt = f"""
    Extract ONLY competitor brand names.

    Product:
    {product}

    Return ONLY a Python list.

    Example:
    ["Sleepwell", "Duroflex"]

    Research:
    {content}
    """

    response = llm.invoke(prompt)

    competitors = ast.literal_eval(
        response.content
    )

    return {
        "competitors": competitors
    }

In [7]:


def product_intelligence_node(state):

    competitors = state["competitors"]

    product = state["product"]

    intelligence_data = {}

    for brand in competitors:

        # DIFFERENT SEARCHES

        cover_search = client.search(

            query=f"""
            {brand} {product} cover GSM
            thread count
            fabric composition
            packaging
            pack size
            """,

            search_depth="advanced",

            max_results=3
        )

        filling_search = client.search(

            query=f"""
            {brand} {product} filling material
            pillow filling
            microfiber
            memory foam
            duck feather
            bamboo filling
            """,

            search_depth="advanced",

            max_results=3
        )

        # COMBINE RESULTS

        content = ""

        for r in cover_search["results"]:
            content += r["content"]

        for r in filling_search["results"]:
            content += r["content"]

        # AI EXTRACTION

        prompt = f"""
        You are a textile product analyst.

        Extract the following information
        for {brand}.

        Required Fields:

        - GSM
        - Thread Count
        - Fabric Composition
        - Filling Type
        - Packaging Type
        - Pack Size

        If information is missing,
        write 'Not Available'.

        Return ONLY a Python dictionary.

        Example:

        {{
            "GSM": "400",
            "Thread Count": "300",
            "Fabric Composition": "Bamboo Viscose",
            "Filling Type": "Memory Foam",
            "Packaging Type": "Vacuum Packed",
            "Pack Size": "Pack of 2"
        }}

        Research Data:
        {content}
        """

        response = llm.invoke(prompt)

        cleaned_response = (
            response.content
            .replace("```python", "")
            .replace("```", "")
            .strip()
        )

        try:

            parsed = ast.literal_eval(
                cleaned_response
            )

            intelligence_data[brand] = parsed

        except Exception as e:

            intelligence_data[brand] = {
                "error": str(e),
                "raw_output": cleaned_response
            }

    return {
        "product_intelligence": intelligence_data
    }

In [8]:


def strategy_node(state):

    intelligence = state[
        "product_intelligence"
    ]

    prompt = f"""
    You are a senior textile market strategist.

    Based on this competitor intelligence,
    generate:

    1. Key Market Trends
    2. Competitor Weaknesses
    3. Premium Product Opportunities
    4. Packaging Recommendations
    5. Differentiation Strategy
    6. Suggested Market Positioning

    Intelligence Data:
    {intelligence}

    Return professional markdown.
    """

    response = llm.invoke(prompt)

    return {
        "strategy_recommendations":
        response.content
    }

In [26]:


graph = StateGraph(MarketState)



graph.add_node(
    "competitor_research",
    competitor_research_node
)

graph.add_node(
    "product_intelligence",
    product_intelligence_node
)

graph.add_node(
    "strategy_generation",
    strategy_node
)

graph.add_node(
    "battlecard_generation",
    battlecard_node
)

# ENTRY POINT

graph.set_entry_point(
    "competitor_research"
)

# FLOW EDGES

graph.add_edge(
    "competitor_research",
    "product_intelligence"
)

graph.add_edge(
    "product_intelligence",
    "strategy_generation"
)

graph.add_edge(
    "strategy_generation",
    "battlecard_generation"
)

graph.add_edge(
    "battlecard_generation",
    END
)

# COMPILE WORKFLOW

workflow = graph.compile()

In [27]:


result = workflow.invoke({

    "product":
    "Bamboo Pillow Cover"

})

print(
    result["battlecard_report"]
)

# Competitor Battlecard Report: Bamboo Pillow Cover
## Executive Summary
The bamboo pillow cover market is characterized by a growing demand for eco-friendly and sustainable products. Our competitor intelligence data reveals that key players such as Cozy Earth, Honeydew, and Cariloha are leveraging bamboo viscose, organic bamboo, and recycled polyester to cater to this trend. However, weaknesses in transparency, product offerings, and quality consistency present opportunities for differentiation and premium product development. This report provides a comprehensive analysis of the market, competitor weaknesses, and recommendations for premium product opportunities, packaging, sourcing, and positioning strategies.

## Competitor Comparison Table
| Competitor | GSM | Thread Count | Fabric Composition | Filling Type | Packaging Type | Pack Size |
| --- | --- | --- | --- | --- | --- | --- |
| Cozy Earth | Not Available | Not Available | Bamboo Viscose | Recycled Polyester, Viscose from Bamb